# 03. Phân chia dữ liệu, kiểm soát rò rỉ và tập tham chiếu SHAP

Giao thức: chia phân tầng 80/20 thành Development/Test; lấy tập SHAP reference 1.000 mẫu từ Development; Core là phần Development còn lại; tạo 5 fold phân tầng **chỉ trên Core**. Test chỉ dùng khi đánh giá cuối, reference chỉ dùng để tính SHAP. Các seed và kích thước được đọc từ `configs/data.yaml` (hiện là 42/42).

Nguồn dữ liệu là CSV gốc UCI tại đường dẫn `raw_data_path` trong cấu hình. **Cần chạy lại notebook này** sau khi thay đổi giao thức; kết quả thực thi cũ đã được xóa.


## 1. Nạp cấu hình và dữ liệu gốc UCI

Dùng cùng nguồn dữ liệu và các hàm chia tập với `scripts/01_make_splits.py` để giữ quy tắc thực thi thống nhất. Chỉ số dòng của dữ liệu gốc được giữ nguyên khi tạo các tập.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'configs' / 'data.yaml').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy configs/data.yaml từ thư mục làm việc hiện tại.')
sys.path.insert(0, str(project_root / 'src'))

from creditrisk.config import load_data_config
from creditrisk.data import load_raw_data
from creditrisk.preprocessing import CreditRiskPreprocessor
from creditrisk.splits import make_core_indices, make_cv_folds, make_outer_split, sample_shap_reference, validate_split_protocol

config = load_data_config(str(project_root / 'configs' / 'data.yaml'))
if not config.splits.stratify:
    raise ValueError('Giao thức Chương 2 yêu cầu chia dữ liệu phân tầng.')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print(f'Cấu hình: test={config.splits.test_size:.0%}, outer seed={config.splits.random_state}, '
      f'reference={config.splits.shap_reference_size}, reference seed={config.splits.shap_reference_seed}, '
      f'folds={config.splits.cv_n_splits}')


In [ ]:
raw_data_path = project_root / config.raw_data_path
df = load_raw_data(raw_data_path, config=config)
target_col = config.columns.target_column
if not df.index.is_unique:
    raise ValueError('Chỉ số dòng dữ liệu phải duy nhất.')

print(f'Dữ liệu gốc: {len(df):,} dòng, {df.shape[1]} cột; tỷ lệ default {df[target_col].mean():.2%}')


## 2. Chia ngoài: Development và Test

Phép chia 80/20 là phân tầng và cố định theo `splits.random_state`. Không dùng Test để chọn siêu tham số, tập tham chiếu hay nghiệm Pareto.


In [ ]:
dev_idx, test_idx = make_outer_split(
    df, target_col=target_col,
    test_size=config.splits.test_size,
    random_state=config.splits.random_state,
)
df_dev, df_test = df.loc[dev_idx], df.loc[test_idx]
assert set(dev_idx).isdisjoint(test_idx)
assert len(df_dev) + len(df_test) == len(df)

split_summary = pd.DataFrame([
    {'Tập': 'Development', 'Số mẫu': len(df_dev), 'Tỷ lệ default': df_dev[target_col].mean()},
    {'Tập': 'Test', 'Số mẫu': len(df_test), 'Tỷ lệ default': df_test[target_col].mean()},
])
display(split_summary)


## 3. Tách SHAP reference khỏi Development

Tập tham chiếu được lấy phân tầng trước khi tạo fold. `Core = Development \ Reference`; reference không tham gia train hoặc validation trong bất kỳ trial nào. Cùng tập reference được dùng để tính global SHAP cho mọi mô hình cần so sánh.


In [ ]:
ref_idx = sample_shap_reference(
    df_dev, target_col=target_col,
    n_samples=config.splits.shap_reference_size,
    random_state=config.splits.shap_reference_seed,
)
core_idx = make_core_indices(dev_idx, ref_idx)
df_core, df_ref = df.loc[core_idx], df.loc[ref_idx]

assert len(df_core) + len(df_ref) == len(df_dev)
assert set(core_idx).isdisjoint(ref_idx)
assert set(core_idx).isdisjoint(test_idx)
assert set(ref_idx).isdisjoint(test_idx)

feature_drop_cols = [target_col, config.columns.id_column]
X_core = df_core.drop(columns=feature_drop_cols, errors='ignore')
y_core = df_core[target_col]
X_ref = df_ref.drop(columns=feature_drop_cols, errors='ignore')
X_test = df_test.drop(columns=feature_drop_cols, errors='ignore')
y_test = df_test[target_col]

partition_summary = pd.DataFrame([
    {'Tập': 'Core', 'Số mẫu': len(df_core), 'Tỷ lệ default': df_core[target_col].mean()},
    {'Tập': 'SHAP reference', 'Số mẫu': len(df_ref), 'Tỷ lệ default': df_ref[target_col].mean()},
    {'Tập': 'Test', 'Số mẫu': len(df_test), 'Tỷ lệ default': df_test[target_col].mean()},
])
display(partition_summary)


## 4. Tạo 5 fold phân tầng trên Core

Cố định cùng các fold cho mọi trial tối ưu đơn mục tiêu và đa mục tiêu. Mỗi quan sát Core nằm trong validation đúng một lần; reference và Test không xuất hiện trong bất kỳ fold nào.


In [ ]:
cv_folds = make_cv_folds(
    df_core, target_col=target_col,
    n_splits=config.splits.cv_n_splits,
    random_state=config.splits.random_state,
)
validated_core_idx = validate_split_protocol(dev_idx, test_idx, cv_folds, ref_idx)
assert list(validated_core_idx) == list(core_idx)

fold_summary = pd.DataFrame([
    {'Fold': fold['fold'] + 1,
     'Train': fold['n_train'],
     'Validation': fold['n_val'],
     'Tỷ lệ default validation': df_core.loc[fold['val_indices'], target_col].mean()}
    for fold in cv_folds
])
display(fold_summary)


## 5. Chuẩn bị tiền xử lý theo từng train fold

Bộ tiền xử lý dưới đây mới chỉ được **khởi tạo**. Khi đánh giá từng fold, tạo một bộ mới và `.fit()` trên train fold, rồi `.transform()` validation và reference tương ứng. Cách này xử lý mã danh mục gốc và giữ tham số tiền xử lý tách khỏi validation/reference/Test. Không `.fit()` trên toàn bộ Development/Core trước cross-validation.


In [ ]:
preprocessor_logistic = CreditRiskPreprocessor(
    scale_numerical=True,
    scaler_type='standard',
    one_hot_categorical=True,
    config=config,
)
print('Đã khởi tạo bộ tiền xử lý cho Logistic Regression; chưa fit dữ liệu.')


## 6. Xuất các tập để kiểm tra

Xuất `core.csv` (gồm nhãn), `test.csv` (gồm nhãn) và `reference_set_1000.csv` (chỉ đặc trưng). Các chỉ số dòng và fold chuẩn cho các script thực nghiệm do `scripts/01_make_splits.py` tạo trong `data/splits`; CSV ở đây phục vụ kiểm tra notebook. Chỉ chạy ô này sau khi các bước kiểm tra giao rỗng ở trên đều thành công.


In [ ]:
target_dir = project_root / config.processed_data_dir
target_dir.mkdir(parents=True, exist_ok=True)

core_out = X_core.assign(**{target_col: y_core})
test_out = X_test.assign(**{target_col: y_test})
core_path = target_dir / 'core.csv'
test_path = target_dir / 'test.csv'
ref_path = target_dir / f'reference_set_{config.splits.shap_reference_size}.csv'

core_out.to_csv(core_path, index=False)
test_out.to_csv(test_path, index=False)
X_ref.to_csv(ref_path, index=False)

print(f'Core: {core_path} ({len(core_out):,} dòng)')
print(f'Test: {test_path} ({len(test_out):,} dòng)')
print(f'SHAP reference: {ref_path} ({len(X_ref):,} dòng)')
